# Notebook 02: Feature Engineering for Demand Forecasting

## CRISP-DM Phase: Data Preparation

**Project**: OptiWMS — AI-Driven Warehouse Management System  
**Reference**: Petropoulos et al. (2022), Sections 2.4 (Feature-based), 2.7 (ML-based)

---

### Objective

Transform raw demand data into ML-ready features following enterprise-grade practices:

1. **Lag features** — capture autoregressive demand patterns
2. **Rolling statistics** — encode local trend and volatility
3. **Calendar features** — month, quarter, year-end effects, Sri Lankan seasonality
4. **Box-Cox / log transforms** — stabilise variance
5. **Scaling** — compare StandardScaler vs MinMax vs RobustScaler
6. **Time-series train/val/test split** — avoid look-ahead bias

### Why This Matters for Warehousing

> *"Feature engineering is the most impactful component of the ML forecasting pipeline."*  
> — M5 competition winners, Petropoulos et al. (2022), Section 2.7.4

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

ROOT = Path('..').resolve()
DATA_DIR = ROOT.parent / 'Forecast model train data optiwms'
GEN_DIR  = ROOT / 'outputs' / 'generated'

fg = pd.read_csv(DATA_DIR / 'hemas_scenario_c_dataset_cleaned.csv')
fg['month'] = pd.to_datetime(fg['month'])
if 'demand_units_clean' in fg.columns:
    fg['demand_units'] = fg['demand_units_clean']
fg = fg.sort_values(['fg_code', 'month']).reset_index(drop=True)

print(f'Loaded FG data: {fg.shape}')
print(f'SKUs: {fg["fg_code"].nunique()}, Months: {fg["month"].nunique()}')

## 1. Calendar / Temporal Features

Warehouse demand is driven by calendar cycles — month-end, quarter-end, and Sri Lankan seasonal events (Sinhala New Year April, Vesak May, Christmas December).

In [ ]:
# 1.1 Basic calendar features
fg['month_num']  = fg['month'].dt.month
fg['quarter']    = fg['month'].dt.quarter
fg['year']       = fg['month'].dt.year
fg['is_year_end'] = fg['month_num'].isin([11, 12]).astype(int)

# Sri Lankan seasonal indicators (Petropoulos 2022, Section 2.2)
SL_PEAK_MONTHS = {4: 'sinhala_new_year', 5: 'vesak', 12: 'christmas', 1: 'new_year'}
fg['is_sl_peak'] = fg['month_num'].isin(SL_PEAK_MONTHS.keys()).astype(int)

# Cyclical encoding (sin/cos) for month — ML models benefit from this
fg['month_sin'] = np.sin(2 * np.pi * fg['month_num'] / 12)
fg['month_cos'] = np.cos(2 * np.pi * fg['month_num'] / 12)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Monthly demand pattern
monthly_avg = fg.groupby('month_num')['demand_units'].mean()
axes[0].bar(monthly_avg.index, monthly_avg.values)
for m in SL_PEAK_MONTHS:
    axes[0].axvline(m, color='red', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Demand')
axes[0].set_title('Average Demand by Month (red = SL peaks)')
axes[0].set_xticks(range(1, 13))

# Cyclical encoding visualisation
theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
axes[1].scatter(np.sin(theta), np.cos(theta), s=100)
for i, m in enumerate(range(1, 13)):
    axes[1].annotate(f'M{m}', (np.sin(theta[i]), np.cos(theta[i])),
                     textcoords='offset points', xytext=(5, 5))
axes[1].set_title('Cyclical Month Encoding (sin/cos)')
axes[1].set_xlabel('sin(2pi * month/12)')
axes[1].set_ylabel('cos(2pi * month/12)')
axes[1].set_aspect('equal')

# Quarterly boxplot
sns.boxplot(data=fg, x='quarter', y='demand_units', ax=axes[2])
axes[2].set_title('Demand by Quarter')

plt.tight_layout()
plt.show()

print(f'New calendar features: month_num, quarter, year, is_year_end, is_sl_peak, month_sin, month_cos')

## 2. Lag Features (Autoregressive)

> *"Lagged values of the target variable capture the serial correlation in time series data."*  
> — Petropoulos et al. (2022), Section 2.7.3

For monthly data, lag-1 captures last month's demand, lag-3 captures quarterly effects, and lag-12 captures yearly seasonality.

In [ ]:
# 2.1 Create lag features per SKU
LAG_PERIODS = [1, 2, 3, 6, 12]

for lag in LAG_PERIODS:
    fg[f'demand_lag_{lag}'] = fg.groupby('fg_code')['demand_units'].shift(lag)

# Display lag feature correlation with target
lag_cols = [f'demand_lag_{l}' for l in LAG_PERIODS]
lag_corr = fg[['demand_units'] + lag_cols].corr().iloc[0, 1:]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Lag correlation bar chart
axes[0].bar(lag_corr.index, lag_corr.values)
axes[0].set_title('Lag Feature Correlation with Demand')
axes[0].set_ylabel('Pearson r')
axes[0].tick_params(axis='x', rotation=30)
for i, (idx, val) in enumerate(lag_corr.items()):
    axes[0].text(i, val + 0.01, f'{val:.3f}', ha='center', fontsize=9)

# Lag-1 scatter plot
sample = fg.dropna(subset=['demand_lag_1']).sample(min(3000, len(fg)), random_state=42)
axes[1].scatter(sample['demand_lag_1'], sample['demand_units'], alpha=0.1, s=5)
axes[1].plot([0, sample['demand_units'].max()], [0, sample['demand_units'].max()],
             'r--', alpha=0.5, label='Perfect persistence')
axes[1].set_xlabel('Demand(t-1)')
axes[1].set_ylabel('Demand(t)')
axes[1].set_title('Lag-1 Scatter (demand persistence)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nLag features created: {lag_cols}')
print(f'Rows with NaN (due to lagging): {fg[lag_cols].isna().any(axis=1).sum()} / {len(fg)}')

## 3. Rolling Window Statistics

Rolling statistics capture local trends and volatility patterns that raw lags miss.

In [ ]:
# 3.1 Rolling mean, std, min, max (windows of 3 and 6 months)
for window in [3, 6]:
    grp = fg.groupby('fg_code')['demand_units']
    fg[f'demand_rmean_{window}'] = grp.transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    fg[f'demand_rstd_{window}']  = grp.transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
    fg[f'demand_rmin_{window}']  = grp.transform(lambda x: x.shift(1).rolling(window, min_periods=1).min())
    fg[f'demand_rmax_{window}']  = grp.transform(lambda x: x.shift(1).rolling(window, min_periods=1).max())

# Coefficient of Variation (rolling)
fg['demand_cv_6'] = fg['demand_rstd_6'] / (fg['demand_rmean_6'] + 1)

# Demand momentum (3-month vs 6-month)
fg['demand_momentum'] = fg['demand_rmean_3'] / (fg['demand_rmean_6'] + 1)

# Visualise rolling features for one SKU
sample_sku = fg.groupby('fg_code')['demand_units'].mean().idxmax()
sku_data = fg[fg['fg_code'] == sample_sku].copy()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(sku_data['month'], sku_data['demand_units'], 'b-o', markersize=4, label='Actual')
axes[0].plot(sku_data['month'], sku_data['demand_rmean_3'], 'r--', label='Rolling Mean (3m)', linewidth=2)
axes[0].plot(sku_data['month'], sku_data['demand_rmean_6'], 'g--', label='Rolling Mean (6m)', linewidth=2)
axes[0].fill_between(sku_data['month'],
                      sku_data['demand_rmean_6'] - sku_data['demand_rstd_6'],
                      sku_data['demand_rmean_6'] + sku_data['demand_rstd_6'],
                      alpha=0.2, color='green', label='6m +/- 1 std')
axes[0].set_title(f'Rolling Features — {sample_sku}')
axes[0].legend()
axes[0].set_ylabel('Demand')

axes[1].plot(sku_data['month'], sku_data['demand_momentum'], 'purple', label='Momentum (3m/6m)')
axes[1].axhline(1.0, color='gray', linestyle='--')
axes[1].set_ylabel('Momentum')
axes[1].set_title('Demand Momentum (>1 = accelerating, <1 = decelerating)')
axes[1].legend()

plt.tight_layout()
plt.show()

roll_cols = [c for c in fg.columns if 'rmean' in c or 'rstd' in c or 'rmin' in c or 'rmax' in c]
print(f'Rolling features created: {len(roll_cols)} features')

## 4. Variance Stabilisation — Box-Cox Transform

> *"Variance stabilisation through transformations (log, Box-Cox) is critical for time series with multiplicative seasonality."*  
> — Petropoulos et al. (2022), Section 2.2.3

In [ ]:
# 4.1 Box-Cox transformation
demand_positive = fg['demand_units'].clip(lower=1)
fg['demand_boxcox'], lambda_bc = stats.boxcox(demand_positive)
fg['demand_log1p'] = np.log1p(fg['demand_units'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(fg['demand_units'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title(f'Original Demand\n(skewness: {fg["demand_units"].skew():.2f})')

axes[1].hist(fg['demand_log1p'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title(f'Log1p Transformed\n(skewness: {fg["demand_log1p"].skew():.2f})')

axes[2].hist(fg['demand_boxcox'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[2].set_title(f'Box-Cox (lambda={lambda_bc:.3f})\n(skewness: {fg["demand_boxcox"].skew():.2f})')

plt.suptitle('Variance Stabilisation Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Optimal Box-Cox lambda: {lambda_bc:.4f}')
print(f'Lambda near 0 -> log transform is optimal')
print(f'Lambda near 1 -> no transform needed')

## 5. Scaling Comparison

Different scaling strategies affect model convergence and performance differently. We compare three standard approaches.

In [ ]:
# 5.1 Scaling comparison
feature_to_scale = fg['demand_units'].dropna().values.reshape(-1, 1)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, scaler) in enumerate(scalers.items()):
    scaled = scaler.fit_transform(feature_to_scale).flatten()
    axes[i].hist(scaled, bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{name}\nRange: [{scaled.min():.2f}, {scaled.max():.2f}]')

plt.suptitle('Scaling Strategy Comparison — Demand', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Scaling recommendation for tree-based models: No scaling needed (LightGBM, XGBoost, CatBoost)')
print('Scaling recommendation for neural nets / linear: StandardScaler or RobustScaler')

## 6. Categorical Encoding

Encode categorical variables for ML models.

In [ ]:
# 6.1 Label encode categorical features
le_cat = LabelEncoder()
fg['fg_category_enc'] = le_cat.fit_transform(fg['fg_category'])

le_sku = LabelEncoder()
fg['fg_code_enc'] = le_sku.fit_transform(fg['fg_code'])

print('Category encoding:')
for i, cls in enumerate(le_cat.classes_):
    print(f'  {cls} -> {i}')

# Note: for tree-based models, label encoding is sufficient
# For linear models, one-hot encoding would be preferred
print(f'\nTotal categories: {len(le_cat.classes_)}')
print(f'Total SKUs: {len(le_sku.classes_)}')

## 7. Time-Series Train / Validation / Test Split

> **Critical**: Time series must be split temporally, not randomly, to avoid look-ahead bias.  
> — Petropoulos et al. (2022), Section 2.7.5

Split: **Train** (first 24 months) / **Validation** (months 25-30) / **Test** (months 31-36)

In [ ]:
# 7.1 Temporal split
sorted_months = sorted(fg['month'].unique())
n_months = len(sorted_months)

train_end = sorted_months[23] if n_months > 23 else sorted_months[int(n_months * 0.66)]
val_end   = sorted_months[29] if n_months > 29 else sorted_months[int(n_months * 0.83)]

train_mask = fg['month'] <= train_end
val_mask   = (fg['month'] > train_end) & (fg['month'] <= val_end)
test_mask  = fg['month'] > val_end

print(f'Train: up to {train_end.strftime("%Y-%m")} ({train_mask.sum():,} rows)')
print(f'Val:   {(train_end + pd.DateOffset(months=1)).strftime("%Y-%m")} to {val_end.strftime("%Y-%m")} ({val_mask.sum():,} rows)')
print(f'Test:  {(val_end + pd.DateOffset(months=1)).strftime("%Y-%m")} onwards ({test_mask.sum():,} rows)')
print(f'\nSplit ratio: {train_mask.mean():.1%} / {val_mask.mean():.1%} / {test_mask.mean():.1%}')

# Visualise the split
fig, ax = plt.subplots(figsize=(16, 4))
monthly_total = fg.groupby('month')['demand_units'].sum()

train_ts = monthly_total[monthly_total.index <= train_end]
val_ts = monthly_total[(monthly_total.index > train_end) & (monthly_total.index <= val_end)]
test_ts = monthly_total[monthly_total.index > val_end]

ax.plot(train_ts.index, train_ts.values, 'b-o', markersize=4, label=f'Train ({len(train_ts)} months)')
ax.plot(val_ts.index, val_ts.values, 'orange', marker='o', markersize=4, label=f'Validation ({len(val_ts)} months)')
ax.plot(test_ts.index, test_ts.values, 'r-o', markersize=4, label=f'Test ({len(test_ts)} months)')
ax.axvline(train_end, color='gray', linestyle='--', alpha=0.5)
ax.axvline(val_end, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Time-Series Split — Total FG Demand')
ax.set_ylabel('Total Demand')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Final Feature Set Summary

In [ ]:
# 8.1 Feature importance preview (correlation with target)
feature_cols = [
    'month_num', 'quarter', 'year', 'is_year_end', 'is_sl_peak',
    'month_sin', 'month_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'demand_lag_6', 'demand_lag_12',
    'demand_rmean_3', 'demand_rmean_6', 'demand_rstd_3', 'demand_rstd_6',
    'demand_rmin_3', 'demand_rmax_3', 'demand_rmin_6', 'demand_rmax_6',
    'demand_cv_6', 'demand_momentum',
    'fg_category_enc', 'fg_code_enc'
]
# Add any original features that exist
for col in ['on_hand_inventory', 'lead_time_days', 'stockout_days', 'supplier_otif',
            'inbound_po_qty', 'open_sales_orders', 'returns_qty',
            'promotion_flag', 'holiday_flag', 'price_per_unit']:
    if col in fg.columns:
        feature_cols.append(col)

available = [c for c in feature_cols if c in fg.columns]

# Correlation with target
corr_with_target = fg[available + ['demand_units']].corr()['demand_units'].drop('demand_units').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(8, len(corr_with_target)*0.3)))
colors = ['green' if v > 0 else 'red' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors, alpha=0.7)
ax.set_xlabel('Pearson Correlation with Demand')
ax.set_title('Feature Correlation with Target Variable')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 8.2 Save engineered dataset
output_dir = Path('..') / 'outputs' / 'engineered'
output_dir.mkdir(parents=True, exist_ok=True)

# Drop rows with NaN from lagging (first 12 months per SKU)
fg_clean = fg.dropna(subset=['demand_lag_12'])
print(f'Rows after dropping lag NaNs: {len(fg_clean):,} (dropped {len(fg) - len(fg_clean):,})')

# Save
fg_clean.to_csv(output_dir / 'fg_features_engineered.csv', index=False)
print(f'Saved to: {output_dir / "fg_features_engineered.csv"}')

print(f'\n=== Feature Engineering Summary ===')
print(f'Total features: {len(available)}')
print(f'  Calendar: 7')
print(f'  Lag: {len(LAG_PERIODS)}')
print(f'  Rolling: {len(roll_cols)}')
print(f'  Encoded: 2 (category, sku)')
print(f'  Original: {len(available) - 7 - len(LAG_PERIODS) - len(roll_cols) - 2}')
print(f'\nDataset shape: {fg_clean.shape}')
print(f'\nNext: Notebook 03 — Model Training')